In [ ]:
#The goal of this project is to classify physiological states (baseline, stress, amusement, and meditation) based on chest sensor signals

In [ ]:
#Outliers
#Outlier analysis was considered. However, since physiological signals naturally contain variations and peaks, no aggressive outlier removal was applied to avoid losing meaningful information.

In [ ]:
import pandas as pd
import os
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
### Loading Dataset

#The WESAD dataset consists of multiple subjects, each containing physiological signals and labels.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/data WESAD.zip'
extract_path = '/content/WESAD_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import os
os.listdir('/content/WESAD_data')

['README.txt', 'LICENSE.txt', 'WESAD']

In [ ]:
os.listdir('/content/WESAD_data/WESAD')

['S13',
 'S6',
 'S17',
 'S15',
 'S4',
 'S2',
 'S11',
 'S10',
 'S9',
 'S5',
 'S7',
 'S8',
 'S3',
 'wesad_readme.pdf',
 'S16',
 'S14']

In [ ]:
os.listdir('/content/WESAD_data/WESAD/S2')

['S2_quest.csv', 'S2_E4_Data', 'S2.pkl', 'S2_respiban.txt', 'S2_readme.txt']

In [ ]:
import os

print(os.listdir('/content'))

['.config', 'drive', 'WESAD_data', 'sample_data']


In [ ]:
import pickle

file_path = '/content/WESAD_data/WESAD/S2/S2.pkl'

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

In [ ]:
data.keys()

dict_keys(['signal', 'label', 'subject'])

In [ ]:
data['signal'].keys()

dict_keys(['chest', 'wrist'])

In [ ]:
data['signal'] ['chest'].keys()

dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])

In [ ]:
data['signal'] ['wrist'].keys()

dict_keys(['ACC', 'BVP', 'EDA', 'TEMP'])

In [ ]:
signals = data['signal']
chest = signals['chest']
labels = data['label']

In [ ]:
wrist = signals['wrist']
subject = data['subject']

In [ ]:
print("Subject:", subject)
print("Labels shape:", labels.shape)

print("\nChest signals shapes:")
for key in chest.keys():
    print(key, chest[key].shape)

print("\nWrist signals shapes:")
for key in wrist.keys():
    print(key, wrist[key].shape)

Subject: S2
Labels shape: (4255300,)

Chest signals shapes:
ACC (4255300, 3)
ECG (4255300, 1)
EMG (4255300, 1)
EDA (4255300, 1)
Temp (4255300, 1)
Resp (4255300, 1)

Wrist signals shapes:
ACC (194528, 3)
BVP (389056, 1)
EDA (24316, 1)
TEMP (24316, 1)


In [ ]:
import numpy as np

print("Unique labels:", np.unique(labels))


Unique labels: [0 1 2 3 4 6 7]


In [ ]:
#(Feature Engineering)
#Windowing + Aggregation
#The raw physiological signals were segmented into fixed-size windows (50 samples per window).Each window was then summarized using the mean value to reduce dimensionality and create meaningful features.

In [ ]:
# 0, 6, 7 not important
#using 1 = baseline , 2 = stress , 3 = amusement , 4 = meditation

In [ ]:
#كل label فيها كام sample
unique, counts = np.unique(labels, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Label {u}: {c}")

Label 0: 2142701
Label 1: 800800
Label 2: 430500
Label 3: 253400
Label 4: 537599
Label 6: 45500
Label 7: 44800


In [ ]:
#ناخد الصفوف 1,2,3,4 ونشيل الباقي مش مهمين
#Data Cleaning & Preparation Steps
#Only relevant labels (1, 2, 3, 4) were kept, while unnecessary labels (0, 6, 7) were removed to focus on meaningful physiological states
valid_labels = [1, 2, 3, 4]

mask = np.isin(labels, valid_labels)

filtered_labels = labels[mask]

In [ ]:
filtered_chest = {}

for key in chest.keys():
    filtered_chest[key] = chest[key][mask]

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'ACC_x': filtered_chest['ACC'][:, 0],
    'ACC_y': filtered_chest['ACC'][:, 1],
    'ACC_z': filtered_chest['ACC'][:, 2],
    'ECG': filtered_chest['ECG'].flatten(),
    'EMG': filtered_chest['EMG'].flatten(),
    'EDA': filtered_chest['EDA'].flatten(),
    'Temp': filtered_chest['Temp'].flatten(),
    'Resp': filtered_chest['Resp'].flatten(),
    'label': filtered_labels.flatten()
})

In [ ]:
#نتأكد إن الداتا نظيفة
#نتأكد إن مفيش labels غريبة
#نكون جاهزين للـ:
#Feature extraction ,windowing
df.shape
df['label'].value_counts()
df.head()

,ACC_x,ACC_y,ACC_z,ECG,EMG,EDA,Temp,Resp,label
0,0.8914,-0.1102,-0.2576,0.030945,-0.003708,5.710983,29.083618,1.191711,1
1,0.8926,-0.1086,-0.2544,0.033646,-0.014145,5.719376,29.122437,1.139832,1
2,0.8930,-0.1094,-0.2580,0.033005,0.010208,5.706406,29.115234,1.141357,1
3,0.8934,-0.1082,-0.2538,0.031815,0.012634,5.712509,29.126709,1.155090,1
4,0.8930,-0.1096,-0.2570,0.030350,0.002060,5.727005,29.100861,1.133728,1


In [ ]:
print(df.shape)
print(df['label'].value_counts())

(2022299, 9)
label
1    800800
4    537599
2    430500
3    253400
Name: count, dtype: int64


In [ ]:
df_sample = (
    df.groupby('label', group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), 25000), random_state=42))
      .reset_index(drop=True)
)

print(df_sample.shape)
print(df_sample['label'].value_counts())

(100000, 9)
label
1    25000
2    25000
3    25000
4    25000
Name: count, dtype: int64


/tmp/ipykernel_38932/3734958239.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 25000), random_state=42))


In [ ]:
df_sample = df_sample.sort_index()

In [ ]:
df_sample.head()

,ACC_x,ACC_y,ACC_z,ECG,EMG,EDA,Temp,Resp,label
0,0.6254,-0.0778,-0.7000,0.069077,0.002380,1.131058,33.982880,2.563477,1
1,0.6190,-0.1038,-0.7174,0.084824,0.003204,3.915787,28.785126,1.573181,1
2,0.6394,-0.0642,-0.7046,0.074570,0.001053,2.215958,28.464569,-1.461792,1
3,0.6258,-0.0632,-0.7178,0.066238,-0.004211,1.905823,28.500305,0.703430,1
4,0.6218,-0.0646,-0.7138,0.009567,0.005447,1.988602,28.406006,-1.039124,1


In [ ]:
# عملت ده علشان لما عملت دروب فوق الترتيب ال مشينا بيه اتلغبط وفي داتا اتشالت ف رجعت عملت group by  و sort
df_sample = df.groupby('label', group_keys=False).sample(n=25000, random_state=42).sort_index()

change_points = df.index[df['label'].ne(df['label'].shift())].tolist()
print(change_points[:20])
#0 → 800800 label = 1
#800800 → 1231300 label = 2
#	1231300 → 1504999	label = 3
#1504999 → 1758399 label = 4

[0, 800800, 1231300, 1504999, 1758399]


In [ ]:
df_sample.shape

(100000, 9)

In [ ]:
#كل 50 صف = window واحدة
#بدل 100000 row هيبقى عندنا 2000 window تقريبًا
window_size = 50

In [ ]:
#كل البيانات اللي label = 1 (baseline
df_part = df.iloc[0:800800]

In [ ]:
df_part.shape
df_part['label'].unique()

array([1], dtype=int32)

In [ ]:
first_window = df_part.iloc[0:50]

In [ ]:
firt_window.shapes

(50, 9)

In [ ]:
first_window.mean(numeric_only=True)

,0
ACC_x,0.888492
ACC_y,-0.102692
ACC_z,-0.248636
ECG,0.058816
EMG,-0.001902
EDA,5.710533
Temp,29.118200
Resp,0.960144
label,1.000000


In [ ]:
second_window = df_part.iloc[50:100]

In [ ]:
second_window.mean(numeric_only=True)

,0
ACC_x,0.893108
ACC_y,-0.101984
ACC_z,-0.248820
ECG,0.073392
EMG,-0.004015
EDA,5.702827
Temp,29.126104
Resp,0.466705
label,1.000000


In [ ]:
windows = []

for i in range(0, len(df_part) - window_size, window_size):
    window = df_part.iloc[i:i+window_size]
    windows.append(window.mean(numeric_only=True))

In [ ]:
len(windows)

16015

In [ ]:
df_windows = pd.DataFrame(windows)

In [ ]:
#خاص بـ label = 1 بس (baseline)
df_windows.head()

,ACC_x,ACC_y,ACC_z,ECG,EMG,EDA,Temp,Resp,label
0,0.888492,-0.102692,-0.248636,0.058816,-0.001902,5.710533,29.118200,0.960144,1.0
1,0.893108,-0.101984,-0.248820,0.073392,-0.004015,5.702827,29.126104,0.466705,1.0
2,0.880876,-0.112720,-0.236324,0.055684,-0.000674,5.692024,29.129850,-0.098358,1.0
3,0.891960,-0.121148,-0.247032,0.080362,-0.002607,5.681900,29.124266,-0.692108,1.0
4,0.879976,-0.105180,-0.271688,0.091602,-0.001769,5.671036,29.124640,-1.230774,1.0


In [ ]:
df_windows['label'].value_counts()

,count
label,
1.0,16015


In [ ]:
#label = 2 ,stress
df_part2 = df.iloc[800800:1231300]
df_part2['label'].unique()

array([2], dtype=int32)

In [ ]:
windows2 = []

for i in range(0, len(df_part2) - window_size, window_size):
    window = df_part2.iloc[i:i+window_size]
    windows2.append(window.mean(numeric_only=True))

In [ ]:
len(windows2)

8609

In [ ]:
df_windows2 = pd.DataFrame(windows2)

In [ ]:
df_windows2['label'].value_counts()

,count
label,
2.0,8609


In [ ]:
#label = 3 → Amusement
df_part3 = df[df['label'] == 3]

In [ ]:
df_part3['label'].unique()

array([3], dtype=int32)

In [ ]:
windows3 = []

for i in range(0, len(df_part3) - window_size, window_size):
    window = df_part3.iloc[i:i+window_size]
    windows3.append(window.mean(numeric_only=True))

In [ ]:
len(windows3)

5067

In [ ]:
df_windows3 = pd.DataFrame(windows3)

In [ ]:
df_windows3['label'].value_counts()

,count
label,
3.0,5067


In [ ]:
#label = 4 → Meditation
df_part4 = df[df['label'] == 4]

In [ ]:
df_part4['label'].unique()

array([4], dtype=int32)

In [ ]:
windows4 = []

for i in range(0, len(df_part4) - window_size, window_size):
    window = df_part4.iloc[i:i+window_size]
    windows4.append(window.mean(numeric_only=True))

In [ ]:
len(windows3)

5067

In [ ]:
df_windows4 = pd.DataFrame(windows4)

In [ ]:
df_windows4['label'].value_counts()

,count
label,
4.0,10751


In [ ]:
# بعد ما جبنا ال label لكل حاجه عندنا هندمجهم كلهم في dataset واحده
df_final = pd.concat([df_windows, df_windows2, df_windows3, df_windows4])

In [ ]:
df_final['label'].value_counts()

,count
label,
1.0,16015
4.0,10751
2.0,8609
3.0,5067


In [ ]:
# لحد هنا خلصنا s2 بكل ال label ال جواها دلوقتي هكرر ال عملته فوق علي باقي ال signals لحد ما نخلصهم

In [ ]:
file_path = '/content/WESAD_data/WESAD/S3/S3.pkl'

In [ ]:
with open(file_path, 'rb') as file:
    data = pickle.load(file, encoding='latin1')

In [ ]:
signals = data['signal']

In [ ]:
chest = signals['chest']

In [ ]:
labels = data['label']

In [ ]:
valid_labels = [1, 2, 3, 4]
mask = np.isin(labels, valid_labels)

In [ ]:
filtered_labels = labels[mask]

In [ ]:
filtered_chest = {}

for key in chest.keys():
    filtered_chest[key] = chest[key][mask]

In [ ]:
df = pd.DataFrame({
    'ACC_x': filtered_chest['ACC'][:, 0],
    'ACC_y': filtered_chest['ACC'][:, 1],
    'ACC_z': filtered_chest['ACC'][:, 2],
    'ECG': filtered_chest['ECG'].flatten(),
    'EMG': filtered_chest['EMG'].flatten(),
    'EDA': filtered_chest['EDA'].flatten(),
    'Temp': filtered_chest['Temp'].flatten(),
    'Resp': filtered_chest['Resp'].flatten(),
    'label': filtered_labels.flatten()
})

In [ ]:
df.shape

(2054501, 9)

In [ ]:
df['label'].value_counts()

,count
label,
1,798000
4,546001
2,448000
3,262500


In [ ]:
change_points = df.index[df['label'].ne(df['label'].shift())].tolist()
print(change_points[:20])

[0, 798000, 1246000, 1527400, 1789900]


In [ ]:
#0: 798000 → label 1
#798000 : 1246000 → label 2
#1246000 : 1527400 → label 3
#1527400 : 1799900 → label 4


In [ ]:
df_part1 = df.iloc[0:798000]
df_part1['label'].unique()

array([1], dtype=int32)

In [ ]:
windows1 = []

for i in range(0, len(df_part1) - window_size, window_size):
    window = df_part1.iloc[i:i+window_size]
    windows1.append(window.mean(numeric_only=True))

In [ ]:
len(windows1)

15959

In [ ]:
df_windows1 = pd.DataFrame(windows1)

In [ ]:
df_windows1['label'].value_counts()

,count
label,
1.0,15959


In [ ]:
df_part2 = df.iloc[798000:1246000]

In [ ]:
windows2 = []

for i in range(0, len(df_part2) - window_size, window_size):
    window = df_part2.iloc[i:i+window_size]
    windows2.append(window.mean(numeric_only=True))

In [ ]:
df_windows2 = pd.DataFrame(windows2)

In [ ]:
df_windows2['label'].value_counts()

,count
label,
2.0,8959


In [ ]:
df_part3 = df[df['label'] == 3]

In [ ]:
windows3 = []

for i in range(0, len(df_part3) - window_size, window_size):
    window = df_part3.iloc[i:i+window_size]
    windows3.append(window.mean(numeric_only=True))

In [ ]:
df_windows3 = pd.DataFrame(windows3)

In [ ]:
df_windows3['label'].value_counts()

,count
label,
3.0,5249


In [ ]:
df_part4 = df[df['label'] == 4]

In [ ]:
windows4 = []

for i in range(0, len(df_part4) - window_size, window_size):
    window = df_part4.iloc[i:i+window_size]
    windows4.append(window.mean(numeric_only=True))

In [ ]:
df_windows4 = pd.DataFrame(windows4)

In [ ]:
df_windows4['label'].value_counts()

,count
label,
4.0,10920


In [ ]:
df_final = pd.concat([df_windows, df_windows2, df_windows3, df_windows4])

In [ ]:
df_final['label'].value_counts()

,count
label,
1.0,16015
4.0,10920
2.0,8959
3.0,5249


In [ ]:
# بدل ما اعد هاعمل s3,s4 ,s5 , s6 لحد الاخر هعمل loop احسن لحد s 17

In [ ]:
print(os.listdir('/content/WESAD_data/WESAD'))

['S13', 'S6', 'S17', 'S15', 'S4', 'S2', 'S11', 'S10', 'S9', 'S5', 'S7', 'S8', 'S3', 'wesad_readme.pdf', 'S16', 'S14']


In [ ]:
subject_folders = sorted([
    folder for folder in os.listdir('/content/WESAD_data/WESAD')
    if folder.startswith('S')
])

print(subject_folders)
print(len(subject_folders))

['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
15


In [ ]:
all_subjects_data = []

valid_labels = [1, 2, 3, 4]
window_size = 50

for subject_id in subject_folders:
    print(f"Processing {subject_id}...")

    file_path = f'/content/WESAD_data/WESAD/{subject_id}/{subject_id}.pkl'

    with open(file_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')

    signals = data['signal']
    chest = signals['chest']
    labels = data['label']

    mask = np.isin(labels, valid_labels)
    filtered_labels = labels[mask]

    filtered_chest = {}
    for key in chest.keys():
        filtered_chest[key] = chest[key][mask]

    df = pd.DataFrame({
        'ACC_x': filtered_chest['ACC'][:, 0],
        'ACC_y': filtered_chest['ACC'][:, 1],
        'ACC_z': filtered_chest['ACC'][:, 2],
        'ECG': filtered_chest['ECG'].flatten(),
        'EMG': filtered_chest['EMG'].flatten(),
        'EDA': filtered_chest['EDA'].flatten(),
        'Temp': filtered_chest['Temp'].flatten(),
        'Resp': filtered_chest['Resp'].flatten(),
        'label': filtered_labels.flatten()
    })

    subject_windows = []

    for label_value in [1, 2, 3, 4]:
        df_part = df[df['label'] == label_value]

        for i in range(0, len(df_part) - 50, 50):
            window = df_part.iloc[i:i+50]
            subject_windows.append(window.mean(numeric_only=True))

    df_subject = pd.DataFrame(subject_windows)
    df_subject['subject'] = subject_id

    print(df_subject.shape)
    print(df_subject['label'].value_counts())

    all_subjects_data.append(df_subject)

df_all = pd.concat(all_subjects_data, ignore_index=True)

print("\nFinal merged dataset shape:", df_all.shape)
print(df_all['subject'].value_counts())

table_all = df_all['label'].value_counts().reset_index()
table_all.columns = ['label', 'count']
table_all

Processing S10...
(43018, 10)
label
1.0    16519
4.0    11143
2.0    10149
3.0     5207
Name: count, dtype: int64
Processing S11...
(42263, 10)
label
1.0    16519
4.0    11074
2.0     9519
3.0     5151
Name: count, dtype: int64
Processing S13...
(42291, 10)
label
1.0    16520
4.0    11129
2.0     9295
3.0     5347
Name: count, dtype: int64
Processing S14...
(42291, 10)
label
1.0    16519
4.0    11115
2.0     9449
3.0     5208
Name: count, dtype: int64
Processing S15...
(42374, 10)
label
1.0    16449
4.0    11115
2.0     9603
3.0     5207
Name: count, dtype: int64
Processing S16...
(42179, 10)
label
1.0    16519
4.0    11087
2.0     9422
3.0     5151
Name: count, dtype: int64
Processing S17...
(42094, 10)
label
1.0    16533
4.0    10233
2.0    10121
3.0     5207
Name: count, dtype: int64
Processing S2...
(40442, 10)
label
1.0    16015
4.0    10751
2.0     8609
3.0     5067
Name: count, dtype: int64
Processing S3...
(41087, 10)
label
1.0    15959
4.0    10920
2.0     8959
3.0     5249
Na

,label,count
0,1.0,246541
1,4.0,165272
2,2.0,139510
3,3.0,78037


In [ ]:
df_all.shape
df_all['subject'].value_counts()

,count
subject,
S10,43018
S15,42374
S13,42291
S14,42291
S11,42263
S16,42179
S5,42150
S17,42094
S8,42066


In [ ]:
df_all.shape

(629360, 10)

In [ ]:
df_all["label"].value_counts(normalize=True)

,proportion
label,
1.0,0.391733
4.0,0.262603
2.0,0.221670
3.0,0.123994


In [ ]:
df_all.isnull().sum()

,0
ACC_x,0
ACC_y,0
ACC_z,0
ECG,0
EMG,0
EDA,0
Temp,0
Resp,0
label,0
subject,0


In [ ]:
df_all.duplicated().sum()
#No duplicate rows were found in the dataset, indicating that each window represents a unique segment of the signal.


np.int64(0)

In [ ]:
df_all.drop_duplicates(inplace=True)

In [ ]:
df_all.info()
#All signal features are stored as numeric (float64), which is suitable for machine learning models.
#The label column represents the class, and the subject column identifies the participant.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 629360 entries, 0 to 629359
Data columns (total 10 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   ACC_x    629360 non-null  float64
 1   ACC_y    629360 non-null  float64
 2   ACC_z    629360 non-null  float64
 3   ECG      629360 non-null  float64
 4   EMG      629360 non-null  float64
 5   EDA      629360 non-null  float64
 6   Temp     629360 non-null  float64
 7   Resp     629360 non-null  float64
 8   label    629360 non-null  float64
 9   subject  629360 non-null  object 
dtypes: float64(9), object(1)
memory usage: 48.0+ MB


In [ ]:
df_all.describe()

,ACC_x,ACC_y,ACC_z,ECG,EMG,EDA,Temp,Resp,label
count,629360.000000,629360.000000,629360.000000,629360.000000,629360.000000,629360.000000,629360.000000,629360.000000,629360.000000
mean,0.779224,-0.031332,-0.382868,0.001124,-0.003011,4.606462,33.876122,0.049911,2.257468
std,0.143222,0.099426,0.293955,0.146355,0.002678,3.358243,1.310729,3.797804,1.224685
min,-5.848452,-5.938480,-5.940408,-1.465841,-1.353056,0.458199,-242.473373,-44.821564,1.000000
25%,0.699835,-0.057704,-0.615550,-0.069209,-0.003882,2.030745,33.600521,-1.917000,1.000000
50%,0.837536,-0.019010,-0.387208,-0.012763,-0.003077,3.694122,34.177872,-0.158157,2.000000
75%,0.891368,0.019816,-0.194192,0.044630,-0.002247,6.498489,34.595901,2.037933,4.000000
max,1.592312,0.387740,0.744232,1.467196,0.026547,20.364708,35.649174,37.638245,4.000000


In [ ]:
#The final dataset contains 629,360 rows after windowing and aggregation, significantly reduced from the original raw signal size.
#(629360 rows,10 columns)

In [ ]:
# Preprocessing for Modeling

In [ ]:
#!find /content -name "S2.pkl"

In [ ]:
#ده لتوحيد مقاييس الأعمدة
# Step 1: Scaling

#from sklearn.preprocessing import StandardScaler

#X = df_model.drop(columns=['label', 'label_name'])
#y = df_model['label']

#import pandas as pd

#df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
#df_scaled['label'] = y.values

#df_scaled.head()

In [ ]:
#Segmentation with window size = 50 , لأن WESAD time-series طويلة جدًا.

#window_size = 50
#step_size = 50

#segments = []
#labels_seg = []

#for i in range(0, len(df_scaled) - window_size, step_size):
    #segment = df_scaled.iloc[i:i + window_size]

    #segments.append(segment.drop(columns=['label']).values)
    #labels_seg.append(segment['label'].mode()[0])

#print("Number of segments:", len(segments))
#print("Number of labels:", len(labels_seg))

In [ ]:
#print(type(segments))
#print(type(labels_seg))
#print(segments[0].shape)
#print(labels_seg[:10])

In [ ]:
#Feature Extraction , كل window فيها 50 صف × 8 أعمدة, وعدد الاعمده هنا 8 مش 9 علشان فوق مكان ف عمود ال label هنا مينفعش علشان ال data هيا ال بتتعلم

#import numpy as np

#features = []

#for seg in segments:
    #feat = []

    #feat.extend(np.mean(seg, axis=0))
    #feat.extend(np.std(seg, axis=0))
    #feat.extend(np.min(seg, axis=0))
    #feat.extend(np.max(seg, axis=0))

    #features.append(feat)

#X_features = np.array(features)
#y_features = np.array(labels_seg)

#print("X_features shape:", X_features.shape)
#print("y_features shape:", y_features.shape)

In [ ]:
#Train Model

#from sklearn.model_selection import train_test_split

#X_train, X_test, y_train, y_test = train_test_split(
 #   X_features, y_features,
  #  test_size=0.2,
   # random_state=42,
    #stratify=y_features)

#print(X_train.shape, X_test.shape)

In [ ]:
# Step 5: Train Random Forest

#from sklearn.ensemble import RandomForestClassifier

#model = RandomForestClassifier(random_state=42)
#model.fit(X_train, y_train)

In [ ]:
# Step 6: Evaluation

#from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
#import pandas as pd
#from IPython.display import display

# Predictions
#y_pred = model.predict(X_test)

# Accuracy
#acc = accuracy_score(y_test, y_pred)

#print(" Model Evaluation")

#print(f"{acc:.4f}")

In [ ]:
#cm = confusion_matrix(y_test, y_pred)

#cm_df = pd.DataFrame(cm,
 #                    index=['Actual Baseline', 'Actual Stress', 'Actual Amusement'],
  #                   columns=['Pred Baseline', 'Pred Stress', 'Pred Amusement'])

#display(cm_df)

In [ ]:
#report = classification_report(y_test, y_pred, output_dict=True)

#report_df = pd.DataFrame(report).transpose()

#display(report_df.round(2))